# Day 9 — Locality Benchmarking

## Goal

* Create a locality-level KPI table that summarizes lice pressure,
compliance, warning signals, and environmental context for use in
PostgreSQL and Grafana.

In [3]:
# Import libraries
import pandas as pd
from pathlib import Path

In [4]:
# Define the feature-file location
feature_file = (
    Path.home()
    / "Documents"
    / "Kazi_Academic"
    / "Projects"
    / "Aquaculture"
    / "fish-health-analytics"
    / "data"
    / "processed"
    / "barentswatch_lice_2025_features.csv"
)

In [5]:
# Load the feature dataset
df = pd.read_csv(feature_file)

/tmp/ipykernel_180679/585919499.py:2: DtypeWarning: Columns (0: weekly_lice_limit) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(feature_file)


In [6]:
# Check dataset size
df.shape

(55718, 45)

In [7]:
# Check available columns
df.columns.tolist()

['week',
 'year',
 'locality_id',
 'locality_name',
 'adult_female_lice',
 'mobile_lice',
 'sessile_lice',
 'probably_without_fish',
 'lice_counted',
 'municipality_id',
 'municipality',
 'county_id',
 'county',
 'latitude',
 'longitude',
 'weekly_lice_limit',
 'above_weekly_lice_limit',
 'sea_temperature_c',
 'production_area_id',
 'production_area',
 'weekly_lice_limit_numeric',
 'compliance_code',
 'adult_female_lice_lag_1',
 'adult_female_lice_lag_2',
 'adult_female_lice_change',
 'adult_female_lice_change_2w',
 'adult_female_lice_mean_3w',
 'distance_to_limit',
 'ratio_to_limit',
 'previous_breach',
 'future_breach',
 'temperature_flag',
 'sea_temperature_clean',
 'high_temperature_check',
 'temp_previous_week',
 'temp_next_week',
 'temp_neighbor_mean',
 'temp_difference',
 'temperature_lag_1',
 'temperature_change',
 'temperature_mean_3w',
 'lice_change_absolute',
 'lice_change_anomaly',
 'anomaly_type',
 'warning_level']

# Count valid lice observations for each locality

In [8]:
# Keep rows where adult female lice was measured
valid_lice = df[
    df["adult_female_lice"].notna()
].copy()

In [9]:
# Group valid observations by locality
locality_groups = valid_lice.groupby(
    ["locality_id", "locality_name"]
)

In [10]:
# Count measured weeks for each locality
valid_observations = locality_groups[
    "adult_female_lice"
].count()

# Calculate lice KPIs

In [11]:
# Average adult female lice by locality
mean_lice = locality_groups[
    "adult_female_lice"
].mean()

In [12]:
# Highest adult female lice value
max_lice = locality_groups[
    "adult_female_lice"
].max()

In [13]:
# Calculate breach rate
# Keep rows where compliance status is known
valid_compliance = df[
    df["compliance_code"].notna()
].copy()

In [14]:
# Group compliance data by locality
compliance_groups = valid_compliance.groupby(
    ["locality_id", "locality_name"]
)

In [15]:
# Calculate breach proportion
breach_rate = compliance_groups[
    "compliance_code"
].mean()

In [16]:
# Convert to percentage
breach_rate_pct = breach_rate * 100

In [17]:
# Calculate environmental context
# Average cleaned sea temperature by locality
mean_temperature = locality_groups[
    "sea_temperature_clean"
].mean()

In [18]:
# Average fraction of the legal lice limit
mean_ratio_to_limit = locality_groups[
    "ratio_to_limit"
].mean()

In [19]:
# Calculate warning frequency
# Start with 0
df["watch_flag"] = 0

In [20]:
# Change Watch rows to 1
df.loc[
    df["warning_level"] == "Watch",
    "watch_flag"
] = 1

In [21]:
# Group by locality
warning_groups = df.groupby(
    ["locality_id", "locality_name"]
)

In [22]:
# Calculate proportion of Watch weeks
watch_rate = warning_groups[
    "watch_flag"
].mean()

In [23]:
# Convert to percentage
watch_rate_pct = watch_rate * 100

# Combine everything into one benchmark table

In [24]:
# ============================================================
# DAY 9 — COMPLETE THE LOCALITY BENCHMARK TABLE
# Starting point:
# valid_observations, mean_lice and max_lice already exist
# ============================================================


# ------------------------------------------------------------
# STEP 1: CALCULATE BREACH RATE
# ------------------------------------------------------------

# Keep only rows where compliance status is known
valid_compliance = df[df["compliance_code"].notna()].copy()

# Group observations by locality
compliance_by_locality = valid_compliance.groupby(
    ["locality_id", "locality_name"]
)

# Calculate the proportion of breach weeks
breach_rate = compliance_by_locality["compliance_code"].mean()

# Convert proportion to percentage
breach_rate_pct = breach_rate * 100



# ------------------------------------------------------------
# STEP 2: CALCULATE WATCH RATE
# ------------------------------------------------------------

# Create a numeric Watch column:
# Normal = 0
# Watch  = 1
df["watch_flag"] = 0

df.loc[
    df["warning_level"] == "Watch",
    "watch_flag"
] = 1

# Group by locality
warning_by_locality = df.groupby(
    ["locality_id", "locality_name"]
)

# Calculate the percentage of weeks classified as Watch
watch_rate = warning_by_locality["watch_flag"].mean()

watch_rate_pct = watch_rate * 100



# ------------------------------------------------------------
# STEP 3: CALCULATE MEAN TEMPERATURE
# ------------------------------------------------------------

# Group by locality
temperature_by_locality = df.groupby(
    ["locality_id", "locality_name"]
)

# Calculate average cleaned sea temperature
mean_temperature = temperature_by_locality[
    "sea_temperature_clean"
].mean()



# ------------------------------------------------------------
# STEP 4: CALCULATE MEAN RATIO TO LEGAL LIMIT
# ------------------------------------------------------------

# Average ratio_to_limit for each locality
mean_ratio_to_limit = temperature_by_locality[
    "ratio_to_limit"
].mean()



# ------------------------------------------------------------
# STEP 5: LOAD DAY 8 RISK ANALYSIS
# ------------------------------------------------------------

# Define processed-data folder
processed_folder = (
    Path.home()
    / "Documents"
    / "Kazi_Academic"
    / "Projects"
    / "Aquaculture"
    / "fish-health-analytics"
    / "data"
    / "processed"
)

# Define Day 8 risk file
risk_file = processed_folder / "day08_risk_analysis.csv"

# Load it
risk_data = pd.read_csv(risk_file)



# ------------------------------------------------------------
# STEP 6: CALCULATE MEAN RISK SCORE
# ------------------------------------------------------------

# Group Day 8 risk data by locality
risk_by_locality = risk_data.groupby(
    ["locality_id", "locality_name"]
)

# Calculate average risk score
mean_risk_score = risk_by_locality[
    "risk_score"
].mean()



# ------------------------------------------------------------
# STEP 7: COUNT HIGH-RISK WEEKS
# ------------------------------------------------------------

# Q4 High was the highest-risk quartile from Day 8

risk_data["high_risk_flag"] = 0

df_high_risk = risk_data["risk_group"] == "Q4 High"

risk_data.loc[
    df_high_risk,
    "high_risk_flag"
] = 1

# Re-group because we added a new column
risk_by_locality = risk_data.groupby(
    ["locality_id", "locality_name"]
)

# Count high-risk weeks for each locality
high_risk_weeks = risk_by_localality = risk_by_locality[
    "high_risk_flag"
].sum()



# ------------------------------------------------------------
# STEP 8: COMBINE ALL LOCALITY KPIs
# ------------------------------------------------------------

# Combine all Series into one table
locality_benchmark = pd.concat(
    [
        valid_observations,
        mean_lice,
        max_lice,
        breach_rate_pct,
        watch_rate_pct,
        mean_temperature,
        mean_ratio_to_limit,
        mean_risk_score,
        high_risk_weeks
    ],
    axis=1
)



# ------------------------------------------------------------
# STEP 9: GIVE THE COLUMNS CLEAR NAMES
# ------------------------------------------------------------

locality_benchmark.columns = [
    "valid_observations",
    "mean_lice",
    "max_lice",
    "breach_rate_pct",
    "watch_rate_pct",
    "mean_temperature_c",
    "mean_ratio_to_limit",
    "mean_risk_score",
    "high_risk_weeks"
]



# ------------------------------------------------------------
# STEP 10: TURN LOCALITY ID AND NAME BACK INTO COLUMNS
# ------------------------------------------------------------

locality_benchmark = locality_benchmark.reset_index()



# ------------------------------------------------------------
# STEP 11: ROUND NUMBERS FOR READABILITY
# ------------------------------------------------------------

locality_benchmark["mean_lice"] = (
    locality_benchmark["mean_lice"].round(3)
)

locality_benchmark["max_lice"] = (
    locality_benchmark["max_lice"].round(3)
)

locality_benchmark["breach_rate_pct"] = (
    locality_benchmark["breach_rate_pct"].round(2)
)

locality_benchmark["watch_rate_pct"] = (
    locality_benchmark["watch_rate_pct"].round(2)
)

locality_benchmark["mean_temperature_c"] = (
    locality_benchmark["mean_temperature_c"].round(2)
)

locality_benchmark["mean_ratio_to_limit"] = (
    locality_benchmark["mean_ratio_to_limit"].round(3)
)

locality_benchmark["mean_risk_score"] = (
    locality_benchmark["mean_risk_score"].round(3)
)



# ------------------------------------------------------------
# STEP 12: INSPECT THE FINAL BENCHMARK TABLE
# ------------------------------------------------------------

locality_benchmark.head(10)



# ------------------------------------------------------------
# STEP 13: LOOK AT THE HIGHEST BREACH-RATE LOCALITIES
# ------------------------------------------------------------

highest_breach_localities = locality_benchmark.sort_values(
    "breach_rate_pct",
    ascending=False
)

highest_breach_localities.head(15)



# ------------------------------------------------------------
# STEP 14: LOOK AT THE HIGHEST MEAN-RISK LOCALITIES
# ------------------------------------------------------------

highest_risk_localities = locality_benchmark.sort_values(
    "mean_risk_score",
    ascending=False
)

highest_risk_localities.head(15)



# ------------------------------------------------------------
# STEP 15: SAVE THE FINAL DAY 9 OUTPUT
# ------------------------------------------------------------

benchmark_file = (
    processed_folder
    / "locality_benchmark_2025.csv"
)

locality_benchmark.to_csv(
    benchmark_file,
    index=False
)

print("Saved:", benchmark_file)
print("Number of localities:", len(locality_benchmark))

Saved: /home/user/Documents/Kazi_Academic/Projects/Aquaculture/fish-health-analytics/data/processed/locality_benchmark_2025.csv
Number of localities: 1082
